# 02.1 — TensorRT Inference on Test Set (ImageNet Val)

Same sweep as `02_tensorrt_inference` but evaluated on the ImageNet validation split (test set).

ONNX exports and TRT engines are assumed to already exist.

Results saved under `resultsv2/test_runs/`.

In [1]:
# ── Toggle dataset here ──────────────────────────────────────────
SPLIT = "val"              # ImageNet val split (= "test" for this project)
OUTPUT_ROOT = "/home/pf4636/code/resnet/quantized_resnets/runs/test_infer"

In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("..").resolve()
PYFILES = PROJECT_ROOT / "pyfiles"
if str(PYFILES) not in sys.path:
    sys.path.insert(0, str(PYFILES))

In [3]:
import json
import time
import torch
import torch.nn as nn
import numpy as np
import pandas as pd

from src.config import ExperimentConfig, with_overrides, set_seed
from src.data import get_dataloader
from src.runner import _get_trt_paths
from trt.trt_infer import trt_evaluate
from utils.utils import load_runs, flatten_runs

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", "{:.3f}".format)
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

torch: 2.10.0+cu130 | cuda: True


In [4]:
CKPT_ROOT = PROJECT_ROOT / "checkpoints"
ONNX_ROOT = PROJECT_ROOT / "onnx"
INPUT_BITS = [8, 4, 2, 1]
TRT_PRECISIONS = ["fp32", "fp16", "int8", "fp8", "int4"]

experiments = {}
for bits in INPUT_BITS:
    ckpt_dir = CKPT_ROOT / f"fp32_{bits}bit"
    if not ckpt_dir.exists():
        continue
    for seed_dir in sorted(ckpt_dir.iterdir()):
        if seed_dir.is_dir() and (seed_dir / "best.pth").exists():
            seed = seed_dir.name
            seed_num = seed.split("_")[-1]
            experiments[(bits, seed)] = {
                "ckpt": str(seed_dir / "best.pth"),
                "seed_num": seed_num,
            }

def make_base(bits, seed, seed_num):
    return ExperimentConfig(
        backend="tensorrt",
        device="cuda",
        batch_size=1,
        seed=42,
        num_eval_batches=None,
        trt_static_shape=True,
        trt_workspace_mb=2048,
        onnx_root=str(ONNX_ROOT / f"fp32_{bits}bit"),
        engine_root=str(PROJECT_ROOT / "engines" / f"fp32_{bits}bit" / seed),
        trt_onnx_prefix=f"resnet_{seed_num}",
        output_root=f"{OUTPUT_ROOT}/fp32_{bits}bit/{seed}",
    )

print("Experiments found:")
for (bits, seed), info in experiments.items():
    print(f"  {bits}bit / {seed}: {info['ckpt']}")

Experiments found:
  8bit / seed_1: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_8bit/seed_1/best.pth
  8bit / seed_2: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_8bit/seed_2/best.pth
  8bit / seed_42: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_8bit/seed_42/best.pth
  4bit / seed_1: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_4bit/seed_1/best.pth
  4bit / seed_2: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_4bit/seed_2/best.pth
  4bit / seed_42: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_4bit/seed_42/best.pth
  2bit / seed_1: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_2bit/seed_1/best.pth
  2bit / seed_2: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_2bit/seed_2/best.pth
  2bit / seed_42: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_2bit/seed_42/best.pth
  1bit / seed_1: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_1bit/seed_1/best

In [5]:
def run_trt_test_experiment(cfg, split=SPLIT):
    cfg = cfg.normalized()
    cfg.validate()
    set_seed(cfg)

    criterion = nn.CrossEntropyLoss()
    loader = get_dataloader(cfg, split=split)

    _, engine_path, _ = _get_trt_paths(cfg)
    if not engine_path.exists():
        raise FileNotFoundError(f"Engine not found: {engine_path}")

    t0 = time.perf_counter()
    tracker = trt_evaluate(engine_path, cfg, loader, criterion)
    elapsed = round(time.perf_counter() - t0, 3)

    payload = {
        "status": "ok",
        "run_id": cfg.run_id(),
        "system": cfg.stamp(),
        "config": cfg.to_dict(),
        "results": tracker.summary(),
        "total_eval_time_sec": elapsed,
        "split": split,
        "error": None,
    }

    out_path = Path(cfg.output_root) / cfg.run_id() / "result.json"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w") as f:
        json.dump(payload, f, indent=2, sort_keys=True)
    print(f"  Saved: {out_path}")

    return payload

## FP32

In [6]:
fp32_records = []
print("TRT Precision: fp32")
for (bits, seed), info in experiments.items():
    base = make_base(bits, seed, info["seed_num"])
    cfg = with_overrides(base, model_precision="fp32", input_quant_bits=bits)

    if cfg.result_json_path().exists():
        print(f"  SKIP {bits}bit / {seed} (result exists)")
        continue

    print(f"\n  {bits}bit / {seed}:")
    payload = run_trt_test_experiment(cfg)
    fp32_records.append(payload)
    r = payload["results"]
    print(f"    top1={r['top1_acc']:.2f}%  top5={r['top5_acc']:.2f}%  infer={r['infer_ms_avg']:.2f}ms")

TRT Precision: fp32

  8bit / seed_1:
[data] Filtered to 5000 samples across 100 classes.
[trt_infer] Engine: /home/pf4636/code/resnet/quantized_resnets/engines/fp32_8bit/seed_1/resnet18_tensorrt_fp32_in8b_cuda_bs1.engine
[trt_infer] Input: 'images'  Output: 'logits'  Dynamic batch: False
[trt_infer] Evaluating 5000 batches (first 30 are warmup) ...
[trt_infer] --- Warmup complete (30 batches) — starting metric collection ---
  [40/5000]  Top-1: 90.00%  Top-5: 100.00%  Infer: 0.96 ms/batch
  [50/5000]  Top-1: 95.00%  Top-5: 100.00%  Infer: 0.94 ms/batch
  [60/5000]  Top-1: 93.33%  Top-5: 100.00%  Infer: 0.93 ms/batch
  [70/5000]  Top-1: 90.00%  Top-5: 97.50%  Infer: 0.93 ms/batch
  [80/5000]  Top-1: 88.00%  Top-5: 98.00%  Infer: 0.95 ms/batch
  [90/5000]  Top-1: 90.00%  Top-5: 98.33%  Infer: 0.95 ms/batch
  [100/5000]  Top-1: 88.57%  Top-5: 98.57%  Infer: 0.94 ms/batch
  [110/5000]  Top-1: 88.75%  Top-5: 98.75%  Infer: 0.94 ms/batch
  [120/5000]  Top-1: 87.78%  Top-5: 97.78%  Infer: 0.

## FP16

In [7]:
fp16_records = []
print("TRT Precision: fp16")
for (bits, seed), info in experiments.items():
    base = make_base(bits, seed, info["seed_num"])
    cfg = with_overrides(base, model_precision="fp16", input_quant_bits=bits)

    if cfg.result_json_path().exists():
        print(f"  SKIP {bits}bit / {seed} (result exists)")
        continue

    print(f"\n  {bits}bit / {seed}:")
    payload = run_trt_test_experiment(cfg)
    fp16_records.append(payload)
    r = payload["results"]
    print(f"    top1={r['top1_acc']:.2f}%  top5={r['top5_acc']:.2f}%  infer={r['infer_ms_avg']:.2f}ms")

TRT Precision: fp16

  8bit / seed_1:
[data] Filtered to 5000 samples across 100 classes.
[trt_infer] Engine: /home/pf4636/code/resnet/quantized_resnets/engines/fp32_8bit/seed_1/resnet18_tensorrt_fp16_in8b_cuda_bs1.engine
[trt_infer] Input: 'images'  Output: 'logits'  Dynamic batch: False
[trt_infer] Evaluating 5000 batches (first 30 are warmup) ...
[trt_infer] --- Warmup complete (30 batches) — starting metric collection ---
  [40/5000]  Top-1: 90.00%  Top-5: 100.00%  Infer: 0.46 ms/batch
  [50/5000]  Top-1: 95.00%  Top-5: 100.00%  Infer: 0.55 ms/batch
  [60/5000]  Top-1: 93.33%  Top-5: 100.00%  Infer: 0.51 ms/batch
  [70/5000]  Top-1: 90.00%  Top-5: 97.50%  Infer: 0.50 ms/batch
  [80/5000]  Top-1: 88.00%  Top-5: 98.00%  Infer: 0.49 ms/batch
  [90/5000]  Top-1: 90.00%  Top-5: 98.33%  Infer: 0.48 ms/batch
  [100/5000]  Top-1: 88.57%  Top-5: 98.57%  Infer: 0.51 ms/batch
  [110/5000]  Top-1: 88.75%  Top-5: 98.75%  Infer: 0.51 ms/batch
  [120/5000]  Top-1: 87.78%  Top-5: 97.78%  Infer: 0.

## INT8

In [8]:
int8_records = []
print("TRT Precision: int8")
for (bits, seed), info in experiments.items():
    base = make_base(bits, seed, info["seed_num"])
    cfg = with_overrides(base, model_precision="int8", input_quant_bits=bits)

    if cfg.result_json_path().exists():
        print(f"  SKIP {bits}bit / {seed} (result exists)")
        continue

    print(f"\n  {bits}bit / {seed}:")
    payload = run_trt_test_experiment(cfg)
    int8_records.append(payload)
    r = payload["results"]
    print(f"    top1={r['top1_acc']:.2f}%  top5={r['top5_acc']:.2f}%  infer={r['infer_ms_avg']:.2f}ms")

TRT Precision: int8

  8bit / seed_1:
[data] Filtered to 5000 samples across 100 classes.
[trt_infer] Engine: /home/pf4636/code/resnet/quantized_resnets/engines/fp32_8bit/seed_1/resnet18_tensorrt_int8_in8b_cuda_bs1.engine
[trt_infer] Input: 'images'  Output: 'logits'  Dynamic batch: False
[trt_infer] Evaluating 5000 batches (first 30 are warmup) ...
[trt_infer] --- Warmup complete (30 batches) — starting metric collection ---
  [40/5000]  Top-1: 90.00%  Top-5: 100.00%  Infer: 0.59 ms/batch
  [50/5000]  Top-1: 95.00%  Top-5: 100.00%  Infer: 0.64 ms/batch
  [60/5000]  Top-1: 93.33%  Top-5: 100.00%  Infer: 0.63 ms/batch
  [70/5000]  Top-1: 90.00%  Top-5: 97.50%  Infer: 0.62 ms/batch
  [80/5000]  Top-1: 88.00%  Top-5: 98.00%  Infer: 0.63 ms/batch
  [90/5000]  Top-1: 90.00%  Top-5: 98.33%  Infer: 0.65 ms/batch
  [100/5000]  Top-1: 90.00%  Top-5: 98.57%  Infer: 0.68 ms/batch
  [110/5000]  Top-1: 90.00%  Top-5: 98.75%  Infer: 0.69 ms/batch
  [120/5000]  Top-1: 88.89%  Top-5: 97.78%  Infer: 0.

## FP8

In [9]:
fp8_records = []
print("TRT Precision: fp8")
for (bits, seed), info in experiments.items():
    base = make_base(bits, seed, info["seed_num"])
    cfg = with_overrides(base, model_precision="fp8", input_quant_bits=bits)

    if cfg.result_json_path().exists():
        print(f"  SKIP {bits}bit / {seed} (result exists)")
        continue

    print(f"\n  {bits}bit / {seed}:")
    payload = run_trt_test_experiment(cfg)
    fp8_records.append(payload)
    r = payload["results"]
    print(f"    top1={r['top1_acc']:.2f}%  top5={r['top5_acc']:.2f}%  infer={r['infer_ms_avg']:.2f}ms")

TRT Precision: fp8

  8bit / seed_1:
[data] Filtered to 5000 samples across 100 classes.
[trt_infer] Engine: /home/pf4636/code/resnet/quantized_resnets/engines/fp32_8bit/seed_1/resnet18_tensorrt_fp8_in8b_cuda_bs1.engine
[trt_infer] Input: 'images'  Output: 'logits'  Dynamic batch: False
[trt_infer] Evaluating 5000 batches (first 30 are warmup) ...
[trt_infer] --- Warmup complete (30 batches) — starting metric collection ---
  [40/5000]  Top-1: 90.00%  Top-5: 100.00%  Infer: 0.61 ms/batch
  [50/5000]  Top-1: 95.00%  Top-5: 100.00%  Infer: 0.60 ms/batch
  [60/5000]  Top-1: 93.33%  Top-5: 100.00%  Infer: 0.65 ms/batch
  [70/5000]  Top-1: 90.00%  Top-5: 97.50%  Infer: 0.64 ms/batch
  [80/5000]  Top-1: 88.00%  Top-5: 98.00%  Infer: 0.63 ms/batch
  [90/5000]  Top-1: 90.00%  Top-5: 98.33%  Infer: 0.63 ms/batch
  [100/5000]  Top-1: 88.57%  Top-5: 98.57%  Infer: 0.62 ms/batch
  [110/5000]  Top-1: 88.75%  Top-5: 98.75%  Infer: 0.61 ms/batch
  [120/5000]  Top-1: 87.78%  Top-5: 97.78%  Infer: 0.63

## INT4

In [10]:
int4_records = []
print("TRT Precision: int4")
for (bits, seed), info in experiments.items():
    base = make_base(bits, seed, info["seed_num"])
    cfg = with_overrides(base, model_precision="int4", input_quant_bits=bits)

    if cfg.result_json_path().exists():
        print(f"  SKIP {bits}bit / {seed} (result exists)")
        continue

    print(f"\n  {bits}bit / {seed}:")
    payload = run_trt_test_experiment(cfg)
    int4_records.append(payload)
    r = payload["results"]
    print(f"    top1={r['top1_acc']:.2f}%  top5={r['top5_acc']:.2f}%  infer={r['infer_ms_avg']:.2f}ms")

TRT Precision: int4

  8bit / seed_1:
[data] Filtered to 5000 samples across 100 classes.
[trt_infer] Engine: /home/pf4636/code/resnet/quantized_resnets/engines/fp32_8bit/seed_1/resnet18_tensorrt_int4_in8b_cuda_bs1.engine
[trt_infer] Input: 'images'  Output: 'logits'  Dynamic batch: False
[trt_infer] Evaluating 5000 batches (first 30 are warmup) ...
[trt_infer] --- Warmup complete (30 batches) — starting metric collection ---
  [40/5000]  Top-1: 90.00%  Top-5: 100.00%  Infer: 0.58 ms/batch
  [50/5000]  Top-1: 95.00%  Top-5: 100.00%  Infer: 0.60 ms/batch
  [60/5000]  Top-1: 93.33%  Top-5: 100.00%  Infer: 0.54 ms/batch
  [70/5000]  Top-1: 90.00%  Top-5: 97.50%  Infer: 0.54 ms/batch
  [80/5000]  Top-1: 88.00%  Top-5: 98.00%  Infer: 0.51 ms/batch
  [90/5000]  Top-1: 90.00%  Top-5: 98.33%  Infer: 0.51 ms/batch
  [100/5000]  Top-1: 88.57%  Top-5: 98.57%  Infer: 0.50 ms/batch
  [110/5000]  Top-1: 86.25%  Top-5: 98.75%  Infer: 0.50 ms/batch
  [120/5000]  Top-1: 85.56%  Top-5: 97.78%  Infer: 0.

## All Results (per seed)

In [11]:
all_rows = []
for bits in INPUT_BITS:
    ckpt_dir = CKPT_ROOT / f"fp32_{bits}bit"
    if not ckpt_dir.exists():
        continue
    for seed_dir in sorted(ckpt_dir.iterdir()):
        if not seed_dir.is_dir():
            continue
        run_dir = f"{OUTPUT_ROOT}/fp32_{bits}bit/{seed_dir.name}"
        runs = load_runs(run_dir, status="ok")
        for r in flatten_runs(runs):
            r["seed"] = seed_dir.name
            r["input_bits_trained"] = bits
            all_rows.append(r)

df = pd.DataFrame(all_rows)
df_trt = df[df["cfg.backend"] == "tensorrt"]

display_cols = [
    "seed", "input_bits_trained", "cfg.model_precision",
    "res.top1_acc", "res.top5_acc", "res.infer_ms_avg", "res.throughput_infer_sps",
]
df_trt[display_cols].sort_values(
    ["input_bits_trained", "seed", "cfg.model_precision"],
    ascending=[False, True, True],
).reset_index(drop=True)

,seed,input_bits_trained,cfg.model_precision,res.top1_acc,res.top5_acc,res.infer_ms_avg,res.throughput_infer_sps
0,seed_1,8,fp16,77.867,93.400,0.470,2125.540
1,seed_1,8,fp32,77.867,93.380,0.954,1048.534
2,seed_1,8,fp8,77.706,93.421,0.627,1596.042
3,seed_1,8,int4,77.706,93.340,0.490,2042.174
4,seed_1,8,int8,77.646,93.280,0.642,1558.384
5,seed_2,8,fp16,78.008,93.843,0.472,2117.554
6,seed_2,8,fp32,78.008,93.863,0.952,1049.882
7,seed_2,8,fp8,78.068,93.763,0.645,1551.314
8,seed_2,8,int4,77.606,93.662,0.497,2013.092
9,seed_2,8,int8,78.189,93.924,0.705,1418.124


## Averaged Results (across seeds)

In [12]:
avg_df = df_trt.groupby(["cfg.backend", "cfg.model_precision", "input_bits_trained"]).agg(
    top1_mean=("res.top1_acc", "mean"),
    top1_std=("res.top1_acc", "std"),
    top5_mean=("res.top5_acc", "mean"),
    top5_std=("res.top5_acc", "std"),
    infer_ms_mean=("res.infer_ms_avg", "mean"),
    infer_ms_std=("res.infer_ms_avg", "std"),
    throughput_mean=("res.throughput_infer_sps", "mean"),
).round(3).reset_index()

avg_df = avg_df.sort_values(
    ["cfg.model_precision", "input_bits_trained"],
    ascending=[True, False],
).reset_index(drop=True)
avg_df

,cfg.backend,cfg.model_precision,input_bits_trained,top1_mean,top1_std,top5_mean,top5_std,infer_ms_mean,infer_ms_std,throughput_mean
0,tensorrt,fp16,8,77.988,0.112,93.588,0.229,0.487,0.027,2058.048
1,tensorrt,fp16,4,78.343,0.360,93.635,0.111,0.472,0.005,2120.904
2,tensorrt,fp16,2,76.868,0.413,93.139,0.060,0.484,0.009,2066.219
3,tensorrt,fp16,1,69.557,0.474,89.470,0.129,0.475,0.002,2105.526
4,tensorrt,fp32,8,77.981,0.103,93.581,0.251,0.955,0.003,1047.124
5,tensorrt,fp32,4,78.343,0.360,93.649,0.101,0.947,0.009,1055.567
6,tensorrt,fp32,2,76.861,0.424,93.132,0.071,0.947,0.012,1056.510
7,tensorrt,fp32,1,69.557,0.469,89.470,0.129,0.941,0.009,1062.943
8,tensorrt,fp8,8,77.941,0.204,93.568,0.176,0.640,0.011,1563.624
9,tensorrt,fp8,4,78.370,0.453,93.642,0.101,0.649,0.027,1543.098


In [13]:
bench_fp32 = PROJECT_ROOT / "results" / "latency_bench" / "tensorrt_fp32_in4b_cuda_bs1.json"
bench_int8 = PROJECT_ROOT / "results" / "latency_bench" / "tensorrt_int8_in8b_cuda_bs1.json"

with open(bench_fp32) as f:
    std_fp = np.std(json.load(f)["latencies_ms"], ddof=1)
with open(bench_int8) as f:
    std_int = np.std(json.load(f)["latencies_ms"], ddof=1)

bench_std_map = {"fp32": std_fp, "fp16": std_fp, "fp8": std_fp, "int8": std_int, "int4": std_int}

avg_df[["cfg.model_precision", "input_bits_trained",
        "top1_mean", "top1_std", "top5_mean", "top5_std",
        "infer_ms_mean"]].assign(
    top1=lambda d: d.apply(lambda r: f"{r['top1_mean']:.2f} ± {r['top1_std']:.2f}", axis=1),
    top5=lambda d: d.apply(lambda r: f"{r['top5_mean']:.2f} ± {r['top5_std']:.2f}", axis=1),
    infer_ms=lambda d: d.apply(lambda r: f"{r['infer_ms_mean']:.3f} ± {bench_std_map[r['cfg.model_precision']]:.3f}", axis=1),
)[["cfg.model_precision", "input_bits_trained", "top1", "top5", "infer_ms"]]

,cfg.model_precision,input_bits_trained,top1,top5,infer_ms
0,fp16,8,77.99 ± 0.11,93.59 ± 0.23,0.487 ± 0.136
1,fp16,4,78.34 ± 0.36,93.64 ± 0.11,0.472 ± 0.136
2,fp16,2,76.87 ± 0.41,93.14 ± 0.06,0.484 ± 0.136
3,fp16,1,69.56 ± 0.47,89.47 ± 0.13,0.475 ± 0.136
4,fp32,8,77.98 ± 0.10,93.58 ± 0.25,0.955 ± 0.136
5,fp32,4,78.34 ± 0.36,93.65 ± 0.10,0.947 ± 0.136
6,fp32,2,76.86 ± 0.42,93.13 ± 0.07,0.947 ± 0.136
7,fp32,1,69.56 ± 0.47,89.47 ± 0.13,0.941 ± 0.136
8,fp8,8,77.94 ± 0.20,93.57 ± 0.18,0.640 ± 0.136
9,fp8,4,78.37 ± 0.45,93.64 ± 0.10,0.649 ± 0.136


In [14]:
out_path = PROJECT_ROOT / "resultsv2" / "test_runs" / "tensorrt_avg_results_test.json"
out_path.parent.mkdir(parents=True, exist_ok=True)
avg_df.to_json(out_path, orient="records", indent=2)
print(f"Saved to {out_path}")

Saved to /home/pf4636/code/resnet/quantized_resnets/resultsv2/test_runs/tensorrt_avg_results_test.json
